Abordagem B: LDiA + LSA + Semantic Frames - NÃO FOI FAVORAVEL

Incorpora verificação por etapa, explicação granular e refinamento R1-R7

In [1]:
# 1. DEPENDÊNCIAS

#%pip install pymupdf pandas openpyxl scikit-learn gensim spacy sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# 2. CONFIGURAÇÃO
from pathlib import Path

PASTA_PADOS   = r"D:\WULDSON\Estudos\Mestrado\IFPB\Disciplinas\orientacao\Transparencia_IA\projeto_anatel\dados\pdfs_anatel"
ARQUIVO_SAIDA = "resultado_requisitos_pados_1b.xlsx"
CHARS_MIN_PAGINA = 100
N_TOPICOS_LDA    = 10   # um tópico por requisito como ponto de partida, aumentado para 10
N_TOPICOS_LSA    = 100  # dimensões da matriz LSA

print(f"Pasta dos PADOs : {Path(PASTA_PADOS).resolve()}")
print(f"Saída           : {Path(ARQUIVO_SAIDA).resolve()}")
print(f"Topicos LDA     : {N_TOPICOS_LDA}")
print(f"Dimensoes LSA   : {N_TOPICOS_LSA}")

Pasta dos PADOs : D:\WULDSON\Estudos\Mestrado\IFPB\Disciplinas\orientacao\Transparencia_IA\projeto_anatel\dados\pdfs_anatel
Saída           : D:\WULDSON\Estudos\Mestrado\IFPB\Disciplinas\orientacao\Transparencia_IA\projeto_mestrado\analises\resultado_requisitos_pados_1b.xlsx
Topicos LDA     : 10
Dimensoes LSA   : 100


In [16]:
# 3. REQUISITOS DE CONFORMIDADE
# Step A - Mesmos requisitos normativos do notebook 1 (Abordagem A)
# R1-R7 definidos com base no RASA, LGT e Lei 9.784/99
# Os padroes regex aqui servem apenas como referencia normativa
# A verificacao na Abordagem B sera feita via LDiA + LSA + SF

REQUISITOS = {
    "R1_Documento_Motivador": {
        "nome": "Documento Motivador Presente",
        "base_legal": "LGT Art. 173",
        "etapa": "Informe ou Relatorio de Fiscalizacao",
        "padroes": [
            r"relat[oó]rio\s+de\s+fiscaliza[cç][aã]o",
            r"auto\s+de\s+infra[cç][aã]o",
            r"informe\s+n[oº°]",
            r"solicita[cç][aã]o\s+de\s+fiscaliza[cç][aã]o",
            r"den[uú]ncia",
            r"processo\s+de\s+fiscaliza[cç][aã]o\s+regulat[oó]ria",
        ],
    },
    "R2_Analise_Previa": {
        "nome": "Analise Previa Realizada",
        "base_legal": "RASA Art. 15",
        "etapa": "Informe",
        "padroes": [
            r"informe\s+n[oº°]\s*\d+",
            r"an[aá]lise\s+(t[eé]cnica|pr[eé]via|dos\s+fatos|da\s+conclus[aã]o)",
            r"ind[íi]cios\s+de\s+(irregularidade|infra[cç][aã]o|descumprimento)",
            r"prop[oõ]e.se\s+a\s+instaurac[aã]o",
            r"descri[cç][aã]o\s+da\s+execu[cç][aã]o",
            r"conclus[aã]o",
        ],
    },
    "R3_Despacho_Instauracao": {
        "nome": "Despacho de Instauracao Formal",
        "base_legal": "Lei 9.784/99",
        "etapa": "Despacho Ordinatorio de Instauracao",
        "padroes": [
            r"despacho\s+\S+\s+de\s+instaurac",
            r"instaurar\s+(procedimento|pado|processo)",
            r"resolve.*instaurar",
            r"auto\s+de\s+infra[cç][aã]o\s+n[oº°]\s*[A-Z]{2}\d+",
            r"instaurou.se\s+o\s+pado",
            r"instaurac[aã]o\s+n[oº°]\s*\d+",
        ],
    },
    "R4_Notificacao_Autuado": {
        "nome": "Notificacao ao Autuado",
        "base_legal": "CF/88 Art. 5 LV",
        "etapa": "Oficio de Notificacao",
        "padroes": [
            r"intima.se\s+(essa|a)\s+prestadora",
            r"[oó]ficio\s+n[oº°].*notifica[cç][aã]o",
            r"notifica[cç][aã]o.*instaurac[aã]o",
            r"prazo.*oferecer\s+defesa",
            r"intima.se.*da.*descri[cç][aã]o\s+do\s+fato",
            r"intimac[aã]o.*fiscalizada",
            r"certid[aã]o.*intimac[aã]o",
            r"tipo\s+de\s+intima[cç][aã]o",
            r"instaurac[aã]o\s+de\s+procedimento",
            r"assunto.*intima[cç][aã]o",
            r"intimac[aã]o\s+para\s+oferecimento",
            r"ao\s+representante\s+legal",
            r"aviso\s+de\s+recebimento",
            r"objeto\s+entregue\s+ao\s+destinat[aá]rio",
            r"yo\d{9}br",
        ],
    },
    "R5_Base_Legal": {
        "nome": "Base Legal Citada",
        "base_legal": "Lei 9.784/99 Art. 50",
        "etapa": "Despacho ou Oficio",
        "padroes": [
            r"art\.\s*\d+.*lgt",
            r"resolu[cç][aã]o\s+n[oº°]\s*\d+",
            r"lei\s+n[oº°]\s*\d+",
            r"normas\s+defini[dt]oras\s+da\s+infra[cç][aã]o",
            r"dispositivos?\s*(normativos?|legais?|relacionados?)",
            r"san[cç][oõ]es?\s+aplic[aá]veis?",
        ],
    },
    "R6_Prazo_Defesa": {
        "nome": "Prazo para Defesa Estabelecido",
        "base_legal": "RASA Art. 33",
        "etapa": "Oficio de Notificacao",
        "padroes": [
            r"prazo\s+de\s+1[05]\s+\((?:quinze|dez)\)\s+dias",
            r"15\s+\(quinze\)\s+dias.*defesa",
            r"oferecer\s+defesa.*prazo",
            r"prazo.*apresentar\s+(defesa|alegac[oõ]es)",
        ],
    },
    "R7_Identificacao_Processo": {
        "nome": "Identificacao Clara do Processo",
        "base_legal": "Lei 9.784/99",
        "etapa": "Qualquer documento",
        "padroes": [
            r"processo\s+n[oº°]\s*\d{5}\.\d{6}/\d{4}-\d{2}",
            r"cnpj\s+n[oº°]\s*\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}",
            r"cpf\s+n[oº°]\s*\d{3}\.\d{3}\.\d{3}-\d{2}",
            r"pado\s+n[oº°]\s*\d{5}\.\d{6}/\d{4}-\d{2}",
        ],
    },
}

print(f"{len(REQUISITOS)} requisitos carregados")
for i, (k, v) in enumerate(REQUISITOS.items(), 1):
    print(f"  R{i} | {v['nome'][:35]:<35} | Etapa: {v['etapa']}")

7 requisitos carregados
  R1 | Documento Motivador Presente        | Etapa: Informe ou Relatorio de Fiscalizacao
  R2 | Analise Previa Realizada            | Etapa: Informe
  R3 | Despacho de Instauracao Formal      | Etapa: Despacho Ordinatorio de Instauracao
  R4 | Notificacao ao Autuado              | Etapa: Oficio de Notificacao
  R5 | Base Legal Citada                   | Etapa: Despacho ou Oficio
  R6 | Prazo para Defesa Estabelecido      | Etapa: Oficio de Notificacao
  R7 | Identificacao Clara do Processo     | Etapa: Qualquer documento


In [17]:
# 4. FUNCOES BASE DE EXTRACAO
# Step B: pre-processamento de texto
# Identico ao notebook 1 — a diferenca da Abordagem B comeca no Step B2 (LDiA)

import re
import fitz
import unicodedata
from pathlib import Path

def extrair_texto_pdf(caminho_pdf):
    try:
        doc = fitz.open(str(caminho_pdf))
        texto = ""
        paginas_vazias = 0
        for pagina in doc:
            t = pagina.get_text()
            if len(t.strip()) < CHARS_MIN_PAGINA:
                paginas_vazias += 1
            else:
                texto += t
        doc.close()
        texto = texto.replace('\xa0', ' ')
        texto = re.sub(r' +', ' ', texto)
        return texto, paginas_vazias
    except Exception:
        return "", 0


def normalizar_texto(texto):
    texto = texto.replace('\xa0', ' ').replace('\n', ' ')
    texto = re.sub(r' +', ' ', texto).lower()
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )


def extrair_sentencas(texto, max_sentencas=50):
    texto = texto.replace('\xa0', ' ').replace('\n', ' ')
    texto = re.sub(r' +', ' ', texto)
    sentencas = re.split(r'(?<=[.!?])\s+', texto)
    sentencas = [s.strip() for s in sentencas if 15 < len(s.strip()) < 500]
    return sentencas[:max_sentencas]


def carregar_corpus():
    """Carrega todo o texto dos 104 PADOs para treinar LDiA e LSA."""
    corpus = []
    pasta_raiz = Path(PASTA_PADOS)
    for pasta in sorted(pasta_raiz.iterdir()):
        if not pasta.is_dir():
            continue
        texto_pado = ""
        for pdf in sorted(pasta.glob("*.pdf")):
            texto, _ = extrair_texto_pdf(pdf)
            texto_pado += texto + " "
        if texto_pado.strip():
            corpus.append({"pado": pasta.name, "texto": texto_pado})
    print(f"Corpus carregado: {len(corpus)} PADOs")
    return corpus


print("Funcoes base carregadas")

Funcoes base carregadas


In [18]:
# 5. STEP B2: LDiA — Latent Dirichlet Allocation
# Substitui detectar_tipo_documento do notebook 1
# LDiA descobre os topicos dominantes de cada documento de forma nao supervisionada
# Os topicos emergem do corpus real dos PADOs — sem supervisao humana

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

# Carrega corpus completo
corpus = carregar_corpus()
textos_corpus = [normalizar_texto(c["texto"]) for c in corpus]

# Stopwords juridicas do dominio PADO
STOPWORDS_PADO = [
    "anatel", "pado", "processo", "procedimento", "instaurar",
    "prestadora", "empresa", "ltda", "sa", "eireli", "brasil", "nacional",
    "agencia", "telecomunicacoes", "art", "lei", "resolucao", "regulamento",
    "nos", "termos", "conforme", "mediante", "referente", "presente",
    "sendo", "tendo", "vista", "face", "razao", "motivo", "objeto",
    "documento", "pagina", "data", "numero", "nro", "sei", "assinado",
    "digitalmente", "verificado", "endereco", "eletronico",
]

# Vetorizador — bag of words sobre o corpus dos PADOs
vetorizador = CountVectorizer(
    max_features=2000,
    min_df=2,
    max_df=0.99,
    stop_words=STOPWORDS_PADO,
    token_pattern=r"[a-zA-ZÀ-ú]{3,}",  # inclui caracteres acentuados
)

matriz_dtm = vetorizador.fit_transform(textos_corpus)
vocabulario = vetorizador.get_feature_names_out()

print(f"Vocabulario: {len(vocabulario)} termos")
print(f"Matriz documento-termo: {matriz_dtm.shape}")

# Treinamento do LDiA
lda_modelo = LatentDirichletAllocation(
    n_components=N_TOPICOS_LDA,
    random_state=42,
    max_iter=20,
    learning_method="batch",
)

lda_modelo.fit(matriz_dtm)
print(f"\nLDiA treinado com {N_TOPICOS_LDA} topicos")

# Exibe os topicos descobertos
print("\nTopicos descobertos pelo LDiA:")
print("-" * 55)
for i, topico in enumerate(lda_modelo.components_):
    top_palavras = [vocabulario[j] for j in topico.argsort()[-10:][::-1]]
    print(f"Topico {i+1}: {', '.join(top_palavras)}")


def classificar_topico_lda(texto):
    """Retorna o topico dominante de um texto."""
    texto_norm = normalizar_texto(texto)
    vetor = vetorizador.transform([texto_norm])
    distribuicao = lda_modelo.transform(vetor)[0]
    topico_dominante = int(np.argmax(distribuicao))
    score = round(float(distribuicao[topico_dominante]), 3)
    return topico_dominante + 1, score, distribuicao


print("\nFuncao classificar_topico_lda pronta")

Corpus carregado: 104 PADOs
Vocabulario: 2000 termos
Matriz documento-termo: (104, 2000)

LDiA treinado com 10 topicos

Topicos descobertos pelo LDiA:
-------------------------------------------------------
Topico 1: servico, como, telefonica, bens, rbr, prazo, infracao, registros, sao, direito
Topico 2: fiscalizacao, produtos, homologacao, multa, infracao, valor, como, prazo, caso, sem
Topico 3: mensal, mbps, fat, servico, valor, fiscalizacao, digital, sarc, cnpj, lon
Topico 4: claro, servico, movel, capitulo, titulo, iii, anexo, bens, secao, pessoal
Topico 5: como, fiscalizacao, sky, servicos, aos, uma, prazo, informe, obrigacoes, intimacao
Topico 6: servico, intimacao, prazo, telefonica, servicos, pelo, obrigacoes, caso, usuarios, usuario
Topico 7: tim, contrato, edp, como, pelo, compartilhamento, prazo, infraestrutura, detentora, ocupante
Topico 8: fiscalizacao, intimacao, prazo, radiodifusao, estacao, entidade, oficio, radio, servico, sistema
Topico 9: mhz, preco, sao, item, lote,

In [19]:
# 6. STEP D2: LSA — Latent Semantic Analysis
# Complementa o Step D do notebook 1 (embeddings sentence-transformers)
# LSA treinada no corpus real dos PADOs captura sinonimia especifica
# do dominio regulatorio brasileiro sem depender de dicionarios externos
# Exemplo esperado: "intimar" retorna "notificar", "comunicar", "cientificar"

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer

# Vetorizador TF-IDF para LSA
tfidf_vectorizer = TfidfVectorizer(
    max_features=3000,
    min_df=2,
    max_df=0.99,
    stop_words=STOPWORDS_PADO,
    token_pattern=r"[a-zA-ZÀ-ú]{3,}",  # inclui caracteres acentuados
    sublinear_tf=True,
)

# Pipeline LSA: TF-IDF + SVD + normalizacao
lsa_pipeline = Pipeline([
    ("tfidf", tfidf_vectorizer),
    ("svd",   TruncatedSVD(n_components=N_TOPICOS_LSA, random_state=42)),
    ("norm",  Normalizer(copy=False)),
])

lsa_pipeline.fit(textos_corpus)

## Diagnóstico 1 — verifica se os termos estão no corpus normalizado
#termos_teste = ["intimar", "fiscalizacao", "instauracao", "defesa", "irregularidade"]
#amostra = textos_corpus[0][:500]
#print("Amostra do corpus normalizado:")
#print(amostra)
#print()
#for termo in termos_teste:
#    ocorrencias = sum(1 for t in textos_corpus if termo in t)
#    print(f"{termo}: aparece em {ocorrencias} PADOs")
#
## Diagnóstico 2 — verifica stopwords e vocabulario
#print()
#for termo in ["intimar", "instauracao"]:
#    na_stop = termo in STOPWORDS_PADO
#    print(f"{termo}: na lista de stopwords = {na_stop}")
#
#vocab_tfidf = tfidf_vectorizer.vocabulary_
#for termo in ["intimar", "instauracao", "fiscalizacao", "defesa"]:
#    no_vocab = termo in vocab_tfidf
#    print(f"{termo}: no vocabulario TF-IDF = {no_vocab}")
#
## Diagnóstico 3 — como intimar aparece no corpus
#import re
#print("\nComo 'intim' aparece no corpus:")
#for i, texto in enumerate(textos_corpus[:10]):
#    matches = re.findall(r"intim\w+", texto)
#    if matches:
#        print(f"  PADO {i}: {matches[:5]}")
    
print(f"LSA treinada: {N_TOPICOS_LSA} dimensoes sobre {len(textos_corpus)} PADOs")

# Variancia explicada
variancia = lsa_pipeline.named_steps["svd"].explained_variance_ratio_
print(f"Variancia explicada acumulada: {variancia.sum():.1%}")


def similaridade_lsa(termo, n=5):
    """Retorna os n termos mais similares a um termo via LSA."""
    # Normaliza o termo antes de buscar no vocabulario
    termo = normalizar_texto(termo)
    vocab = tfidf_vectorizer.vocabulary_
    if termo not in vocab:
        return []
    idx = vocab[termo]
    componentes = lsa_pipeline.named_steps["svd"].components_
    vetor_termo = componentes[:, idx]
    scores = componentes.T.dot(vetor_termo)
    indices_top = scores.argsort()[::-1][1:n+1]
    termos = tfidf_vectorizer.get_feature_names_out()
    return [(termos[i], round(float(scores[i]), 3)) for i in indices_top]

# Mapeamento de formas canonicas para formas presentes no corpus
FORMAS_CORPUS = {
    "intimar": "intimacao",
    "notificar": "notificacao",
    "instaurar": "instauracao",
    "fiscalizar": "fiscalizacao",
    "autuar": "autuado",
}

def enriquecer_sentenca_lsa(sentenca, n_sinonimos=3):
    sentenca_norm = normalizar_texto(sentenca)
    tokens = re.findall(r"[a-zA-ZÀ-ú]{3,}", sentenca_norm)
    termos_enriquecidos = set(tokens)
    for token in tokens:
        # Substitui forma canonica pela forma presente no corpus
        token_busca = FORMAS_CORPUS.get(token, token)
        similares = similaridade_lsa(token_busca, n=n_sinonimos)
        for termo, score in similares:
            if score > 0.3:
                termos_enriquecidos.add(termo)
    return " ".join(termos_enriquecidos)



def enriquecer_sentenca_lsa(sentenca, n_sinonimos=3):
    """Adiciona sinonimos LSA aos termos da sentenca para enriquecer o matching."""
    # Normaliza antes de tokenizar — alinha com o vocabulario construido sem acentos
    sentenca_norm = normalizar_texto(sentenca)
    tokens = re.findall(r"[a-zA-Z]{3,}", sentenca_norm)
    termos_enriquecidos = set(tokens)
    for token in tokens:
        similares = similaridade_lsa(token, n=n_sinonimos)
        for termo, score in similares:
            if score > 0.3:
                termos_enriquecidos.add(termo)
    return " ".join(termos_enriquecidos)


# Teste de sinonimia no dominio PADO
print("\nTeste de sinonimia LSA no corpus dos PADOs:")
print("-" * 45)
for termo_teste in ["intimacao", "fiscalizacao", "instauracao", "defesa", "irregularidade"]:
    similares = similaridade_lsa(termo_teste, n=5)
    if similares:
        print(f"{termo_teste}: {', '.join(f'{t}({s})' for t, s in similares)}")
    else:
        print(f"{termo_teste}: nao encontrado no vocabulario")

print("\nLSA pronta")

LSA treinada: 100 dimensoes sobre 104 PADOs
Variancia explicada acumulada: 99.9%

Teste de sinonimia LSA no corpus dos PADOs:
---------------------------------------------
intimacao: tacito(0.012), util(0.011), dia(0.011), peticionamento(0.011), primeiro(0.011)
fiscalizacao: sfi(0.02), regional(0.017), relatorio(0.016), regulatoria(0.012), fiscalizada(0.012)
instauracao: rodrigo(0.018), melo(0.015), coge(0.015), auxiliar(0.015), apuracao(0.014)
defesa: instauracao(0.008), descumprimento(0.008), prazo(0.008), artigo(0.008), como(0.007)
irregularidade: cpae(0.012), ebc(0.011), rgc(0.01), tim(0.009), qsa(0.008)

LSA pronta


In [7]:
# 7. STEP C + STEP D2: Semantic Frames enriquecidos com LSA
# Diferenca em relacao ao notebook 1:
# As sentencas de referencia de cada frame sao enriquecidas via LSA
# antes de gerar os embeddings — isso amplia a cobertura semantica
# com vocabulario especifico do corpus dos PADOs

import spacy
from sentence_transformers import SentenceTransformer, util

nlp = spacy.load("pt_core_news_sm")
modelo_st = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

FRAMES_REQUISITOS = {
    "R1_Documento_Motivador": {
        "nome": "Documento Motivador Presente",
        "base_legal": "LGT Art. 173",
        "etapa_esperada": ["Informe", "Relatorio de Fiscalizacao", "Auto de Infracao"],
        "verbos_nucleo": ["constatar", "verificar", "identificar", "apurar", "fiscalizar"],
        "objetos_nucleo": ["irregularidade", "indicio", "infracao", "descumprimento"],
        "sentencas_referencia": [
            "A fiscalizacao constatou indicios de irregularidade cometida pela prestadora.",
            "O relatorio de fiscalizacao identificou descumprimento de obrigacoes.",
            "O auto de infracao registra a irregularidade verificada em campo.",
            "A denuncia aponta irregularidade no funcionamento da estacao.",
        ],
    },
    "R2_Analise_Previa": {
        "nome": "Analise Previa Realizada",
        "base_legal": "RASA Art. 15",
        "etapa_esperada": ["Informe"],
        "verbos_nucleo": ["propor", "recomendar", "concluir", "analisar", "instaurar"],
        "objetos_nucleo": ["instauracao", "procedimento", "pado", "apuracao"],
        "sentencas_referencia": [
            "Propoe-se a instauracao de Procedimento para Apuracao de Descumprimento de Obrigacoes.",
            "Diante do exposto propoe-se instaurar o PADO em desfavor da prestadora.",
            "A analise conclui pela instauracao do procedimento sancionador.",
            "O informe recomenda a abertura de processo administrativo sancionador.",
        ],
    },
    "R3_Despacho_Instauracao": {
        "nome": "Despacho de Instauracao Formal",
        "base_legal": "Lei 9.784/99",
        "etapa_esperada": ["Despacho de Instauracao", "Despacho Ordinatorio"],
        "verbos_nucleo": ["instaurar", "determinar", "resolver", "autuar", "expedir"],
        "objetos_nucleo": ["procedimento", "pado", "processo", "instauracao"],
        "sentencas_referencia": [
            "O Gerente resolve instaurar Procedimento para Apuracao de Descumprimento de Obrigacoes.",
            "Determina a instauracao de procedimento de apuracao em desfavor da prestadora.",
            "O Despacho Ordinatorio de Instauracao formaliza a abertura do PADO.",
            "Instaura-se o PADO para apuracao das irregularidades identificadas.",
        ],
    },
    "R4_Notificacao_Autuado": {
        "nome": "Notificacao ao Autuado",
        "base_legal": "CF/88 Art. 5 LV",
        "etapa_esperada": ["Oficio de Notificacao", "Certidao de Intimacao"],
        "verbos_nucleo": ["intimar", "notificar", "comunicar", "expedir", "cientificar"],
        "objetos_nucleo": ["intimacao", "notificacao", "defesa", "prestadora", "autuado"],
        "sentencas_referencia": [
            "Expedir intimacao a prestadora para que ofereca defesa no prazo de 15 dias.",
            "Intima-se essa Prestadora da instauracao do Pado para apresentar defesa.",
            "Notifica-se o autuado para ciencia da instauracao do procedimento.",
            "O oficio comunica a prestadora da abertura do processo sancionador.",
            "Nos termos do artigo 82 inciso II do Regimento Interno da Anatel intima-se essa Prestadora.",
            "Tipo de Intimacao Instauracao de Procedimento Documento Principal da Intimacao Oficio.",
            "Aviso de Recebimento destinatario representante legal prestadora.",
            "Certidao de cumprimento da intimacao eletronica referente a instauracao de procedimento.",
        ],
    },
    "R5_Base_Legal": {
        "nome": "Base Legal Citada",
        "base_legal": "Lei 9.784/99 Art. 50",
        "etapa_esperada": ["Despacho de Instauracao", "Oficio de Notificacao"],
        "verbos_nucleo": ["citar", "fundamentar", "dispor", "prever", "estabelecer"],
        "objetos_nucleo": ["artigo", "resolucao", "lei", "regulamento", "sancao"],
        "sentencas_referencia": [
            "As sancoes previstas no artigo 3 do Regulamento de Aplicacao de Sancoes Administrativas.",
            "Fundamentado no artigo 173 da Lei Geral de Telecomunicacoes.",
            "Nos termos do artigo 82 do Regimento Interno da Anatel.",
            "Com base no dispositivo legal do Regulamento Geral de Direitos do Consumidor.",
        ],
    },
    "R6_Prazo_Defesa": {
        "nome": "Prazo para Defesa Estabelecido",
        "base_legal": "RASA Art. 33",
        "etapa_esperada": ["Oficio de Notificacao"],
        "verbos_nucleo": ["conceder", "estabelecer", "fixar", "determinar", "oferecer"],
        "objetos_nucleo": ["prazo", "dias", "defesa", "alegacoes", "manifestacao"],
        "sentencas_referencia": [
            "Conceder prazo de 15 quinze dias para apresentacao de defesa.",
            "No prazo de 15 dias ofereca defesa e produza as provas que julgar cabiveis.",
            "Fica estabelecido o prazo de quinze dias para manifestacao do autuado.",
            "A prestadora tem 15 dias para apresentar suas alegacoes e provas.",
            "prazo de 15 quinze dias contados a partir do recebimento para oferecer defesa.",
            "para que no prazo de 15 quinze dias ofereca defesa e produza as provas que julgar cabiveis.",
            "o interessado tem prazo para apresentar defesa administrativa e produzir provas.",
        ],
    },
    "R7_Identificacao_Processo": {
        "nome": "Identificacao Clara do Processo",
        "base_legal": "Lei 9.784/99",
        "etapa_esperada": ["Qualquer documento"],
        "verbos_nucleo": ["identificar", "numerar", "registrar", "indicar"],
        "objetos_nucleo": ["processo", "pado", "cnpj", "interessado", "prestadora"],
        "sentencas_referencia": [
            "Processo numero identificado com dados completos do autuado.",
            "O PADO e instaurado em desfavor da prestadora CNPJ identificado.",
            "Identificado o processo com numero SEI e dados completos da autuada.",
        ],
    },
}

# Enriquece sentencas de referencia com LSA antes de gerar embeddings
print("Enriquecendo frames com LSA e computando embeddings...")
for req_id, frame in FRAMES_REQUISITOS.items():
    sentencas_enriquecidas = []
    for s in frame["sentencas_referencia"]:
        s_enriquecida = s + " " + enriquecer_sentenca_lsa(s, n_sinonimos=3)
        sentencas_enriquecidas.append(s_enriquecida)
    frame["sentencas_enriquecidas"] = sentencas_enriquecidas
    frame["embeddings_referencia"] = modelo_st.encode(
        sentencas_enriquecidas,
        convert_to_tensor=True,
    )
    print(f"  {req_id}: {len(sentencas_enriquecidas)} sentencas enriquecidas")

print("\nFrames prontos com enriquecimento LSA")
print(f"Threshold de matching: 0.55")
print(f"Modelo: paraphrase-multilingual-MiniLM-L12-v2")

c:\Users\WDS\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Enriquecendo frames com LSA e computando embeddings...
  R1_Documento_Motivador: 4 sentencas enriquecidas
  R2_Analise_Previa: 4 sentencas enriquecidas
  R3_Despacho_Instauracao: 4 sentencas enriquecidas
  R4_Notificacao_Autuado: 8 sentencas enriquecidas
  R5_Base_Legal: 4 sentencas enriquecidas
  R6_Prazo_Defesa: 7 sentencas enriquecidas
  R7_Identificacao_Processo: 3 sentencas enriquecidas

Frames prontos com enriquecimento LSA
Threshold de matching: 0.55
Modelo: paraphrase-multilingual-MiniLM-L12-v2


In [21]:
# 8. STEP E + ENTREGAS DOS RESULTADOS
# Entrega 1: verificacao por etapa processual
# Entrega 2: explicacao granular do nao conforme
# Entrega 3: refinamento via LDiA — tipo de documento identificado por topico

def verificar_requisito_semantico_lsa(req_id, frame, texto_total, threshold=0.55):
    """Matching semantico com sentencas enriquecidas por LSA."""
    sentencas = extrair_sentencas(texto_total)
    if not sentencas:
        return "NAO", 0.0, "sem sentencas extraiveis"
    embeddings_doc = modelo_st.encode(sentencas, convert_to_tensor=True)
    scores = util.cos_sim(frame["embeddings_referencia"], embeddings_doc)
    score_max = float(scores.max())
    idx_doc = int(scores.argmax() % len(sentencas))
    sentenca_match = sentencas[idx_doc] if idx_doc < len(sentencas) else ""
    resultado = "SIM" if score_max >= threshold else "NAO"
    return resultado, round(score_max, 3), sentenca_match[:200]


def detectar_etapas_presentes(documentos):
    """Identifica quais etapas processuais estao presentes na pasta do PADO."""
    etapas = set()
    for doc in documentos:
        tipo = doc["tipo_lda"]
        etapas.add(tipo)
    return etapas


def classificar_tipo_documento_lda(texto):
    """Classifica o tipo de documento via topico dominante do LDiA."""
    topico, score, distribuicao = classificar_topico_lda(texto)

    # Mapeamento de topico para tipo de documento
    # Sera ajustado apos inspecao dos topicos descobertos na celula 5
    MAPA_TOPICOS = {
        1: "Despacho de Instauracao",
        2: "Outro",
        3: "Outro",
        4: "Auto de Infracao",
        5: "Informe",
        6: "Oficio de Notificacao",
        7: "Certidao de Intimacao",
    }
    return MAPA_TOPICOS.get(topico, "Outro"), score


def explicar_nao_conforme(req_id, req_info, frame, documentos, flag_ocr, paginas_vazias):
    """
    Entrega 2 do orientador: explica por que um requisito retornou NAO.
    Retorna uma string descritiva da causa raiz.
    """
    etapas_presentes = {doc["tipo_lda"] for doc in documentos}
    etapas_esperadas = set(frame.get("etapa_esperada", []))

    # Causa 1: PDF escaneado
    if flag_ocr == "SIM" and paginas_vazias > 0:
        return f"PDF com {paginas_vazias} paginas sem texto extraivel — possivel documento escaneado"

    # Causa 2: etapa processual ausente
    etapas_faltando = etapas_esperadas - etapas_presentes
    if etapas_faltando:
        return f"Etapa processual ausente na pasta: {', '.join(etapas_faltando)}"

    # Causa 3: etapa presente mas requisito nao encontrado no texto
    return f"Documento da etapa '{', '.join(etapas_esperadas)}' presente mas requisito nao localizado no texto"


def verificar_requisito_por_etapa(req_id, req_info, frame, documentos, threshold=0.55):
    """
    Entrega 1 do orientador: verifica o requisito no documento
    da etapa correta, nao no texto completo do PADO.
    """
    etapas_esperadas = set(frame.get("etapa_esperada", []))

    # Filtra documentos da etapa esperada
    docs_etapa = [d for d in documentos if d["tipo_lda"] in etapas_esperadas]

    if not docs_etapa:
        # Fallback: usa texto completo se etapa nao encontrada
        texto_total = " ".join(d["texto"] for d in documentos)
        res, score, sentenca = verificar_requisito_semantico_lsa(
            req_id, frame, texto_total, threshold
        )
        return res, score, sentenca, "fallback — etapa nao identificada"

    # Verifica apenas nos documentos da etapa correta
    melhor_score = 0.0
    melhor_sentenca = ""
    melhor_etapa = ""

    for doc in docs_etapa:
        res, score, sentenca = verificar_requisito_semantico_lsa(
            req_id, frame, doc["texto"], threshold
        )
        if score > melhor_score:
            melhor_score = score
            melhor_sentenca = sentenca
            melhor_etapa = doc["tipo_lda"]

    resultado = "SIM" if melhor_score >= threshold else "NAO"
    return resultado, round(melhor_score, 3), melhor_sentenca, melhor_etapa


print("Funcoes de verificacao por etapa e explicacao granular prontas")

Funcoes de verificacao por etapa e explicacao granular prontas


In [9]:
# 9. ANALISE PRINCIPAL
# Integra LDiA + LSA + SF + verificacao por etapa + explicacao granular

import pandas as pd

def analisar_pado_1b(pasta_pado):
    """Analisa todos os PDFs de uma pasta de PADO — Abordagem B."""
    arquivos = sorted(Path(pasta_pado).glob("*.pdf"))
    if not arquivos:
        return None

    # Extracao de texto e classificacao LDiA por documento
    documentos = []
    total_paginas_vazias = 0
    for pdf in arquivos:
        texto, pag_vazias = extrair_texto_pdf(pdf)
        tipo_lda, score_lda = classificar_tipo_documento_lda(texto)
        documentos.append({
            "arquivo": pdf.name,
            "texto": texto,
            "tipo_lda": tipo_lda,
            "score_lda": score_lda,
        })
        total_paginas_vazias += pag_vazias

    textos = [d["texto"] for d in documentos]
    texto_meta = " ".join(textos)

    # Metadados
    processo = re.search(r"\d{5}\.\d{6}/\d{4}-\d{2}", texto_meta)
    processo = processo.group() if processo else "N/A"

    autuado = re.search(r"interessado[:\s]+([^\n]{5,60})", texto_meta, re.IGNORECASE)
    autuado = autuado.group(1).strip()[:60] if autuado else "N/A"

    # Tipos de documento identificados pelo LDiA
    tipos_lda = ", ".join(sorted(set(d["tipo_lda"] for d in documentos)))

    # Topico dominante do PADO como um todo (via LDiA)
    texto_total_norm = normalizar_texto(" ".join(textos))
    topico_pado, score_pado, _ = classificar_topico_lda(texto_total_norm)

    flag_ocr = "SIM" if total_paginas_vazias > 0 else "NAO"

    resultado = {
        "PADO":              Path(pasta_pado).name,
        "Processo":          processo,
        "Autuado":           autuado,
        "Topico_LDA":        topico_pado,
        "Score_LDA":         score_pado,
        "Tipos_Docs_LDA":    tipos_lda,
        "Qtd_PDFs":          len(arquivos),
        "paginas_sem_texto": total_paginas_vazias,
        "flag_ocr":          flag_ocr,
    }

    # Verificacao por requisito com etapa + explicacao granular
    total_sim = 0
    for req_id, req_info in REQUISITOS.items():
        frame = FRAMES_REQUISITOS[req_id]

        res, score, sentenca, etapa_usada = verificar_requisito_por_etapa(
            req_id, req_info, frame, documentos
        )

        # Explicacao granular do NAO conforme
        if res == "NAO":
            explicacao = explicar_nao_conforme(
                req_id, req_info, frame, documentos, flag_ocr, total_paginas_vazias
            )
        else:
            explicacao = f"Encontrado na etapa: {etapa_usada} | score: {score}"

        resultado[req_id]                      = res
        resultado[f"{req_id}_score"]           = score
        resultado[f"{req_id}_etapa"]           = etapa_usada
        resultado[f"{req_id}_explicacao"]      = explicacao
        resultado[f"{req_id}_sentenca"]        = sentenca[:200] if sentenca else ""

        if res == "SIM":
            total_sim += 1

    resultado["Conformidade"] = f"{total_sim}/7"
    resultado["Pct"]          = round(total_sim / 7 * 100)
    return resultado


# Execucao nos 104 PADOs
pasta_raiz = Path(PASTA_PADOS)
subpastas  = sorted([p for p in pasta_raiz.iterdir() if p.is_dir()])

print(f"{len(subpastas)} pastas encontradas\n")
print("Processando Abordagem B (LDiA + LSA + SF)...\n")

resultados_1b = []
for pasta in subpastas:
    r = analisar_pado_1b(pasta)
    if r:
        resultados_1b.append(r)
        print(f"{r['PADO'][:45]} | {r['Conformidade']} | Topico LDA: {r['Topico_LDA']}")

print(f"\n{len(resultados_1b)} PADOs analisados — Abordagem B")

104 pastas encontradas

Processando Abordagem B (LDiA + LSA + SF)...

53000.062585_2010-06 | 5/7 | Topico LDA: 8
53500.007555_2026-55 | 6/7 | Topico LDA: 6
53500.008890_2026-71 | 7/7 | Topico LDA: 5
53500.008991_2025-61 | 6/7 | Topico LDA: 6
53500.008995_2025-49 | 4/7 | Topico LDA: 6
53500.009858_2026-11 | 6/7 | Topico LDA: 6
53500.011614_2022-66 | 6/7 | Topico LDA: 5
53500.014139_2023-61 | 7/7 | Topico LDA: 1
53500.016622_2024-61 | 7/7 | Topico LDA: 5
53500.017655_2022-66 | 7/7 | Topico LDA: 4
53500.019425_2026-65 | 7/7 | Topico LDA: 5
53500.021979_2025-41 | 6/7 | Topico LDA: 6
53500.022299_2024-64 | 7/7 | Topico LDA: 1
53500.030037_2023-92 | 7/7 | Topico LDA: 5
53500.034731_2025-41 | 6/7 | Topico LDA: 6
53500.036351_2018-11 | 7/7 | Topico LDA: 6
53500.036487_2023-99 | 7/7 | Topico LDA: 5
53500.047569_2022-88 | 5/7 | Topico LDA: 5
53500.057042_2024-23 | 7/7 | Topico LDA: 6
53500.057071_2024-95 | 7/7 | Topico LDA: 6
53500.068455_2024-33 | 6/7 | Topico LDA: 6
53500.083357_2023-45 | 6/7 

In [12]:
# 11. SALVAR RESULTADO DA ABORDAGEM B EM EXCEL
from openpyxl.styles import PatternFill, Font, Alignment

def limpar_para_excel(valor):
    """Remove caracteres invalidos para o Excel."""
    if not isinstance(valor, str):
        return valor
    # Remove caracteres de controle e caracteres invalidos para Excel
    valor = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', valor)
    return valor


ARQUIVO_SAIDA_1B = "resultado_requisitos_pados_1b.xlsx"

# Limpa todos os campos string do resultado
resultados_1b_limpos = []
for r in resultados_1b:
    r_limpo = {k: limpar_para_excel(v) for k, v in r.items()}
    resultados_1b_limpos.append(r_limpo)

cols_resumo = [
    "PADO", "Processo", "Autuado", "Topico_LDA", "Score_LDA",
    "Tipos_Docs_LDA", "Conformidade", "Pct",
    "paginas_sem_texto", "flag_ocr"
] + list(REQUISITOS.keys())

df_resumo = pd.DataFrame(resultados_1b_limpos)[cols_resumo]
df_detalhe = pd.DataFrame(resultados_1b_limpos)

with pd.ExcelWriter(ARQUIVO_SAIDA_1B, engine='openpyxl') as writer:
    df_resumo.to_excel(writer, sheet_name='Matriz de Conformidade', index=False)
    df_detalhe.to_excel(writer, sheet_name='Detalhes e Evidencias', index=False)

    ws = writer.sheets['Matriz de Conformidade']
    azul     = PatternFill(start_color='2E5090', end_color='2E5090', fill_type='solid')
    verde    = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
    vermelho = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
    amarelo  = PatternFill(start_color='FFEB9C', end_color='FFEB9C', fill_type='solid')

    for cell in ws[1]:
        cell.fill = azul
        cell.font = Font(color='FFFFFF', bold=True)
        cell.alignment = Alignment(horizontal='center', wrap_text=True)

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            if cell.value == 'SIM':
                cell.fill = verde
                cell.alignment = Alignment(horizontal='center')
            elif cell.value == 'NAO':
                cell.fill = vermelho
                cell.alignment = Alignment(horizontal='center')

    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = min(
            max(len(str(c.value or '')) for c in col) + 4, 35
        )

print(f"Planilha salva em: {ARQUIVO_SAIDA_1B}")

Planilha salva em: resultado_requisitos_pados_1b.xlsx


In [13]:
# 10. COMPARACAO ABORDAGEM A vs ABORDAGEM B
import pandas as pd

# Carrega os dois resultados
df_a = pd.read_excel("resultado_requisitos_pados.xlsx", sheet_name="Matriz de Conformidade")
df_b = pd.read_excel("resultado_requisitos_pados_1b.xlsx", sheet_name="Matriz de Conformidade")

# Seleciona apenas colunas relevantes para comparacao
cols_req = list(REQUISITOS.keys())
df_a_comp = df_a[["PADO", "Pct"] + cols_req].copy()
df_b_comp = df_b[["PADO", "Pct"] + cols_req].copy()

# Cruza pelo PADO
df_comp = df_a_comp.merge(df_b_comp, on="PADO", suffixes=("_A", "_B"))

# Diferenca de conformidade
df_comp["Diferenca_Pct"] = df_comp["Pct_B"] - df_comp["Pct_A"]
df_comp["Vencedor"] = df_comp["Diferenca_Pct"].apply(
    lambda x: "A" if x < 0 else ("B" if x > 0 else "Igual")
)

# Estatisticas gerais
total = len(df_comp)
a_vence = (df_comp["Vencedor"] == "A").sum()
b_vence = (df_comp["Vencedor"] == "B").sum()
iguais  = (df_comp["Vencedor"] == "Igual").sum()
media_a = df_comp["Pct_A"].mean()
media_b = df_comp["Pct_B"].mean()
a_7 = (df_comp["Pct_A"] == 100).sum()
b_7 = (df_comp["Pct_B"] == 100).sum()

print("=" * 60)
print("COMPARACAO ABORDAGEM A vs ABORDAGEM B")
print("=" * 60)
print(f"Total de PADOs comparados : {total}")
print(f"PADOs 7/7 — Abordagem A   : {a_7} ({a_7/total*100:.1f}%)")
print(f"PADOs 7/7 — Abordagem B   : {b_7} ({b_7/total*100:.1f}%)")
print(f"Media conformidade A       : {media_a:.1f}%")
print(f"Media conformidade B       : {media_b:.1f}%")
print(f"Abordagem A superior       : {a_vence} PADOs ({a_vence/total*100:.1f}%)")
print(f"Abordagem B superior       : {b_vence} PADOs ({b_vence/total*100:.1f}%)")
print(f"Resultado igual            : {iguais} PADOs ({iguais/total*100:.1f}%)")

# Comparacao por requisito
print("\nConcordancia por requisito (A == B):")
print("-" * 45)
for req_id in cols_req:
    col_a = req_id + "_A"
    col_b = req_id + "_B"
    concordancia = (df_comp[col_a] == df_comp[col_b]).sum()
    pct = concordancia / total * 100
    print(f"  {req_id[:30]:<30} {concordancia}/{total} ({pct:.1f}%)")

# PADOs onde B supera A
print("\nPADOs onde Abordagem B supera A:")
print("-" * 45)
df_b_melhor = df_comp[df_comp["Vencedor"] == "B"][["PADO", "Pct_A", "Pct_B", "Diferenca_Pct"]]
print(df_b_melhor.to_string(index=False))

COMPARACAO ABORDAGEM A vs ABORDAGEM B
Total de PADOs comparados : 104
PADOs 7/7 — Abordagem A   : 87 (83.7%)
PADOs 7/7 — Abordagem B   : 18 (17.3%)
Media conformidade A       : 94.2%
Media conformidade B       : 74.1%
Abordagem A superior       : 72 PADOs (69.2%)
Abordagem B superior       : 9 PADOs (8.7%)
Resultado igual            : 23 PADOs (22.1%)

Concordancia por requisito (A == B):
---------------------------------------------
  R1_Documento_Motivador         67/104 (64.4%)
  R2_Analise_Previa              99/104 (95.2%)
  R3_Despacho_Instauracao        34/104 (32.7%)
  R4_Notificacao_Autuado         95/104 (91.3%)
  R5_Base_Legal                  54/104 (51.9%)
  R6_Prazo_Defesa                83/104 (79.8%)
  R7_Identificacao_Processo      77/104 (74.0%)

PADOs onde Abordagem B supera A:
---------------------------------------------
                PADO  Pct_A  Pct_B  Diferenca_Pct
53512.001695_2011-02     71    100             29
53554.000104_2002-94     43     86            

In [23]:
# SALVAR MODELO LDiA PARA USO NO NOTEBOOK 3
import pickle

# MAPA_TOPICOS extraido para fora da funcao para ser salvo
MAPA_TOPICOS_GLOBAL = {
    1: "Despacho de Instauracao",
    2: "Outro",
    3: "Outro",
    4: "Auto de Infracao",
    5: "Informe",
    6: "Oficio de Notificacao",
    7: "Certidao de Intimacao",
    8: "Outro",
    9: "Outro",
    10: "Outro",
}

artefatos_lda = {
    "lda_modelo": lda_modelo,
    "vetorizador": vetorizador,
    "MAPA_TOPICOS": MAPA_TOPICOS_GLOBAL,
    "N_TOPICOS_LDA": N_TOPICOS_LDA,
    "vocabulario": list(vetorizador.vocabulary_.keys()),
}

with open("modelo_lda_pados.pkl", "wb") as f:
    pickle.dump(artefatos_lda, f)

print("Modelo LDiA salvo em: modelo_lda_pados.pkl")
print(f"Topicos: {N_TOPICOS_LDA}")
print(f"Vocabulario: {len(vetorizador.vocabulary_)} termos")

Modelo LDiA salvo em: modelo_lda_pados.pkl
Topicos: 10
Vocabulario: 2000 termos
